# Hipotez 6 Colab Demo

Bu notebook, **Liquid Neural Network / Neural Circuit Policy ile güvenli mobil robot navigasyonu** projesini Google Colab üzerinde denemek için hazırlandı.

Notebook üç amaç için kullanılabilir:

1. Repoyu Colab ortamına almak ve bağımlılıkları kurmak.
2. Kısa bir 2D navigasyon simülasyonu çalıştırıp PNG/GIF çıktılarını göstermek.
3. İsteğe bağlı olarak küçük bir CfC/LTC ablation deneyi çalıştırmak.

> Repo private ise ilk kurulum hücresi senden GitHub token isteyebilir. Token ekrana basılmaz ve notebook dosyasına kaydedilmez.

## 1. Kurulum

Private repo için GitHub'da fine-grained token oluştururken en azından ilgili repoya **Contents: Read-only** yetkisi vermen yeterlidir. Repo public yapılırsa token alanını boş bırakabilirsin.

In [ ]:
from pathlib import Path
import getpass
import os
import subprocess
import sys

REPO_URL = "https://github.com/heimdilon/hypothesis-6-lnn-neurosymbolic.git"
REPO_DIR = Path("/content/hypothesis-6-lnn-neurosymbolic")
PROJECT_DIR = REPO_DIR / "hypothesis_6_lnn_neurosymbolic"

def run(cmd, cwd=None, display_cmd=None):
    if display_cmd is None:
        display_cmd = " ".join(map(str, cmd))
    print("$", display_cmd)
    subprocess.run(list(map(str, cmd)), cwd=cwd, check=True)

if not REPO_DIR.exists():
    token = os.environ.get("GITHUB_TOKEN")
    if token is None:
        token = getpass.getpass("Private repo için GitHub token gir; repo public ise Enter'a bas: ").strip()
    clone_url = REPO_URL
    if token:
        clone_url = REPO_URL.replace("https://", f"https://{token}@")
    run(
        ["git", "clone", "--depth", "1", clone_url, REPO_DIR],
        display_cmd=f"git clone --depth 1 {REPO_URL} {REPO_DIR}",
    )
else:
    print(f"Repo zaten var: {REPO_DIR}")

os.chdir(PROJECT_DIR)
print("Çalışma klasörü:", Path.cwd())

run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"])
print("Kurulum tamam.")

## 2. Hızlı Simülasyon

Bu hücre zorlu harita setinde kısa bir `CfC residual` denemesi çalıştırır. Çıktılar `results/` ve `figures/` klasörlerine yazılır.

In [ ]:
import subprocess
import sys

cmd = [
    sys.executable,
    "src/run_lnn_experiment.py",
    "--episodes", "2",
    "--scenario-set", "hard",
    "--liquid-backend", "cfc",
    "--tag", "colab_cfc_smoke",
    "--ncp-hidden", "16",
    "--ncp-sparsity", "0.50",
    "--ncp-baseline-scale", "1.0",
    "--ncp-residual-scale", "0.35",
    "--ncp-learning-rate", "0.003",
]
subprocess.run(cmd, check=True)

## 3. Sonuçları Göster

Aşağıdaki hücre özet CSV dosyasını ve üretilen başarı/çarpışma grafiğini gösterir.

In [ ]:
from IPython.display import Image, display
from pathlib import Path
import pandas as pd

summary_csv = Path("results/summary_results_colab_cfc_smoke.csv")
figure_png = Path("figures/h6_success_collision_colab_cfc_smoke.png")

display(pd.read_csv(summary_csv))
display(Image(filename=str(figure_png)))

## 4. Repodaki Hazır Görseller

Ablation grafiği ve labirent GIF'i repoda hazır durur. Sunumda hızlı göstermek için bu hücreyi çalıştırabilirsin.

In [ ]:
from IPython.display import Image, display

display(Image(filename="figures/ncp_ablation_success_collision.png"))
display(Image(filename="figures/h6_2d_real_ncp_cfc_labyrinth.gif"))

## 5. Colab İçinde Arayüzü Aç

Bu hücre yerel Python arayüz sunucusunu Colab kernel portu üzerinden açar. Colab dışındaki bir ortamda çalıştırıyorsan tarayıcıdan `http://127.0.0.1:8765` adresine gidebilirsin.

In [ ]:
import subprocess
import sys
import time

PORT = 8765

if "server_proc" in globals() and server_proc.poll() is None:
    print("Server zaten çalışıyor.")
else:
    server_proc = subprocess.Popen([
        sys.executable,
        "src/custom_map_server.py",
        "--host", "0.0.0.0",
        "--port", str(PORT),
    ])
    time.sleep(3)
    print("Server başlatıldı.")

try:
    from google.colab import output
    output.serve_kernel_port_as_iframe(PORT, height=850)
except Exception as exc:
    print(f"Colab iframe açılamadı: {exc}")
    print(f"Yerel URL: http://127.0.0.1:{PORT}")

## 6. Arayüz Sunucusunu Durdur

Arayüzle işin bitince bu hücreyi çalıştırabilirsin.

In [ ]:
if "server_proc" in globals() and server_proc.poll() is None:
    server_proc.terminate()
    server_proc.wait(timeout=10)
    print("Server durduruldu.")
else:
    print("Çalışan server yok.")

## 7. Opsiyonel: Kısa Ablation Deneyi

Bu hücre tam bilimsel sonuçların küçük bir Colab versiyonunu üretir. Daha uzun sürer. Sunum demosu için şart değildir.

In [ ]:
import subprocess
import sys

cmd = [
    sys.executable,
    "src/train_ncp_ablation.py",
    "--train-sequences", "12",
    "--val-sequences", "4",
    "--seq-len", "16",
    "--imitation-epochs", "2",
    "--batch-size", "4",
    "--rl-episodes", "2",
    "--rl-max-steps", "50",
    "--eval-episodes", "1",
    "--eval-max-steps", "80",
    "--hidden-dim", "16",
]
subprocess.run(cmd, check=True)

In [ ]:
from IPython.display import Image, display
import pandas as pd

display(pd.read_csv("results/ncp_ablation_group_summary.csv"))
display(Image(filename="figures/ncp_ablation_success_collision.png"))